# FTIR ML Solver v3 — Colab Training

## Before running
1. **Runtime → Change runtime type → GPU (T4)**
2. Run cells top to bottom

In [ ]:
# Cell 1 — Download repo and install
import urllib.request, zipfile, os, shutil, sys

url = 'https://github.com/DrSuppe/FTIR_absorbtion_ML/archive/refs/heads/main.zip'
print('Downloading...')
urllib.request.urlretrieve(url, '/content/repo.zip')

print('Extracting...')
with zipfile.ZipFile('/content/repo.zip') as z:
    z.extractall('/content/')

extracted = '/content/FTIR_absorbtion_ML-main'
target    = '/content/ftir'
if os.path.exists(target):
    shutil.rmtree(target)
os.rename(extracted, target)
os.chdir(target)
print('Working directory:', os.getcwd())

# Install WITHOUT overwriting Colab's pre-installed torch/numpy
# (do NOT use requirements-pinned.txt — it has versions that conflict with Colab)
!pip install . --no-deps -q

# Colab kernels don't always refresh sys.path immediately after pip install.
# Adding src/ to sys.path guarantees it imports immediately.
sys.path.insert(0, '/content/ftir/src')

# Verify
from ftir_analysis.constants import DEFAULT_TARGET_SPECIES
print('ftir_analysis installed OK')
print('Target species:', DEFAULT_TARGET_SPECIES)

In [ ]:
# Cell 2 — Build / Rebuild Manifest
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f"Manifest: {len(m)} spectra, {m.species.nunique()} unique species")

In [ ]:
# Cell 3 — A/B Run A (~12 min, conservative anti-collapse)
!python3 train.py \
    --n-synthetic 5000 \
    --epochs 12 \
    --batch-size 64 \
    --reference-weight 0.05 \
    --active-label-weight 1.5 \
    --inactive-label-weight 1.0 \
    --warmup-epochs 1.5 \
    --checkpoint-dir outputs/checkpoints/run_a \
    --reports-dir outputs/reports/run_a \
    --log-every 2

In [ ]:
# Cell 4 — A/B Run B (~12 min, stronger positive emphasis)
!python3 train.py \
    --n-synthetic 5000 \
    --epochs 12 \
    --batch-size 64 \
    --reference-weight 0.10 \
    --active-label-weight 2.0 \
    --inactive-label-weight 1.0 \
    --warmup-epochs 1.5 \
    --checkpoint-dir outputs/checkpoints/run_b \
    --reports-dir outputs/reports/run_b \
    --log-every 2

In [ ]:
# Cell 5 — Compare A/B winners by best val mean log-MAE (tie: more species beating zero baseline)
import torch
from pathlib import Path

ckpt_a = Path('outputs/checkpoints/run_a/best_model.pt')
ckpt_b = Path('outputs/checkpoints/run_b/best_model.pt')

if not ckpt_a.exists() or not ckpt_b.exists():
    print('Run both A and B first.')
else:
    a = torch.load(ckpt_a, map_location='cpu')
    b = torch.load(ckpt_b, map_location='cpu')
    a_metric = float(a.get('val_log_mae_mean', float('inf')))
    b_metric = float(b.get('val_log_mae_mean', float('inf')))
    a_beat = int(a.get('species_beating_zero_baseline', -1))
    b_beat = int(b.get('species_beating_zero_baseline', -1))

    print(f'Run A: val_log_mae_mean={a_metric:.4f}, species_beating_zero={a_beat}/11')
    print(f'Run B: val_log_mae_mean={b_metric:.4f}, species_beating_zero={b_beat}/11')

    if abs(a_metric - b_metric) < 1e-9:
        winner = 'A' if a_beat >= b_beat else 'B'
        reason = 'tie on log-MAE, tie-broken by species beating zero baseline'
    else:
        winner = 'A' if a_metric < b_metric else 'B'
        reason = 'lower best val mean log-MAE'

    print(f'Winner: Run {winner} ({reason})')

In [ ]:
# Cell 6 — View the latest per-species MAE plot
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.image as mpimg

plot_dirs = [
    Path('outputs/reports/run_a'),
    Path('outputs/reports/run_b'),
    Path('outputs/reports'),
]
plots = []
for d in plot_dirs:
    if d.exists():
        plots.extend(d.glob('mae_per_species_*.png'))
plots = sorted(plots, key=lambda p: p.stat().st_mtime)
if plots:
    img = mpimg.imread(plots[-1])
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'MAE — {plots[-1].parent.name}/{plots[-1].name}')
    plt.show()
else:
    print('No MAE plots yet — run training first.')

In [ ]:
# Cell 7 — Download best checkpoint
from pathlib import Path
import torch
from google.colab import files

ckpt_a = Path('outputs/checkpoints/run_a/best_model.pt')
ckpt_b = Path('outputs/checkpoints/run_b/best_model.pt')

if ckpt_a.exists() and ckpt_b.exists():
    a = torch.load(ckpt_a, map_location='cpu')
    b = torch.load(ckpt_b, map_location='cpu')
    a_metric = float(a.get('val_log_mae_mean', float('inf')))
    b_metric = float(b.get('val_log_mae_mean', float('inf')))
    if a_metric <= b_metric:
        winner = ckpt_a
        label = 'A'
    else:
        winner = ckpt_b
        label = 'B'
    print(f'Downloading winner Run {label}: {winner}')
    files.download(str(winner))
elif ckpt_a.exists():
    print(f'Downloading Run A checkpoint: {ckpt_a}')
    files.download(str(ckpt_a))
elif ckpt_b.exists():
    print(f'Downloading Run B checkpoint: {ckpt_b}')
    files.download(str(ckpt_b))
else:
    raise FileNotFoundError('No run_a/run_b checkpoint found. Run training first.')